In [0]:
!pip install feature_engine

In [0]:
!pip install xgboost

In [0]:
!pip install lightgbm

In [0]:
!pip install catboost

In [0]:
import numpy as np
import pandas as pd

from feature_engine import imputation

import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn import model_selection
from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

from mlflow.models import infer_signature

import os
import logging

In [0]:
# SAMPLE
df_abt = spark.table("workspace.tmw_trabalhofinal.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef']==df_abt['dtRef'].max()]

df_train_test = df_abt[df_abt['dtRef']!=df_abt['dtRef'].max()]

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
# Somente features pontos
#features = df_train_test.columns.tolist()[2:-11]

X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")


In [0]:
X_train.head()

In [0]:
# EXPLORE

is_na = X_train.isna().mean()
is_na[is_na>0]

# 0 (zero):
# qtdCursosIniciados
# qtdCursosFinalizados
# github_2025
# python_2025
# sql_2025
# pandas_2025
# estatistica_2025
# qtEpisodiosConcluidos28d
# pctTendenciaFrequencia28d_56d
# pctTendenciaPontos28d_56d
# pctTendenciaPontosPositivos28d_56d
# pctTendenciaTransacoes28d_56d

# Valor máximo da base:
# DiasUltimoStreak
# avgTempoInicioUltimo
# avgTempoInicioFim
# avgDiasEntreTransacoes

In [0]:
scaler = MinMaxScaler()

In [0]:
# MODIFY

features_0 = [
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'github_2025',
    'python_2025',
    'sql_2025',
    'pandas_2025',
    'estatistica_2025',
    'qtEpisodiosConcluidos28d',
    'pctTendenciaFrequencia28d_56d',
    'pctTendenciaPontos28d_56d',
    'pctTendenciaPontosPositivos28d_56d',
    'pctTendenciaTransacoes28d_56d'
]

imput_0 = imputation.ArbitraryNumberImputer(
        variables=features_0, 
        arbitrary_number=0
)

imput_max_tail = imputation.EndTailImputer(
        variables=[
                'DiasUltimoStreak',
                'avgTempoInicioUltimo',
                'avgTempoInicioFim', 
                'avgDiasEntreTransacoes'],
        imputation_method='max', 
        fold=1
)

In [0]:
# AJUSTE DOS MÉTODOS
imput_0.fit(X_train)
imput_max_tail.fit(X_train)

# # APLICAÇÃO DOS MÉTODOS
X_train_transform = imput_0.transform(X_train)
X_train_transform = imput_max_tail.transform(X_train_transform)

is_na = X_train_transform.isna().mean()
is_na[is_na>0]

In [0]:
# df_train = X_train.copy()
# df_train[target] = y_train.copy()
# 
# df_train[features] = df_train[features].astype(float)
# 
# bivariada = df_train.groupby(target)[features].median().T
# bivariada['ratio'] = (bivariada[1] + 0.001) / (bivariada[0]+0.001)
# bivariada = bivariada.sort_values(by='ratio', ascending=False)
# bivariada

df_train = X_train_transform.copy()
df_train[target] = y_train
sumario = df_train.groupby(target)[features].mean().T
sumario['rate'] = sumario[1] / sumario[0]
sumario = sumario.sort_values(by='rate', ascending=False)

df_features_excluir = sumario[sumario['rate'].between(0.9, 1.1)]
print(df_features_excluir)

df_features_manter = sumario[(sumario['rate'] < 0.9) | (sumario['rate'] > 1.1)]
features_manter = df_features_manter.index.tolist()
print(features_manter)

In [0]:
seletor_colunas = ColumnTransformer(
    transformers=[("features", "passthrough", features_manter)],
    remainder="drop"
)

In [0]:
df_train

In [0]:
# Regressão Logística

pipeline_logistica = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    ('seletor_colunas', seletor_colunas),
    ('scaler', scaler),
    ('modelo', LogisticRegression(max_iter=3000, random_state=42))
])

grid_logistica = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
    "modelo__solver": ["liblinear", "lbfgs"]
}


In [0]:
# Regressão Logística com Seleção de Variáveis

cv_selecao = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

modelo_selecao = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

seletor_stepwise = SequentialFeatureSelector(
    estimator=modelo_selecao,
    n_features_to_select='auto',
    direction="forward",
    scoring="roc_auc",
    cv=cv_selecao,
    n_jobs=-1
)

pipeline_logistica_selecao = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    ('seletor_colunas', seletor_colunas),
    ('scaler', scaler),
    ('seletor', seletor_stepwise),
    ('modelo', LogisticRegression(max_iter=3000, random_state=42))
])

grid_logistica_selecao = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
    "modelo__solver": ["liblinear", "lbfgs"]
}


In [0]:
# Random Forest

pipeline_random_forest = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ("modelo", RandomForestClassifier(random_state=42, n_jobs=-1))
])

# grid_random_forest = {
    # "modelo__n_estimators": [300, 600, 1000],
    # "modelo__max_depth": [None, 3, 5, 8, 15],
    # "modelo__min_samples_leaf": [3, 5, 10, 20],
    # "modelo__max_features": ["sqrt", 0.5],
    # "modelo__class_weight": [None, "balanced"]
# }

grid_random_forest = {
    "modelo__n_estimators": [350, 1000],
    "modelo__max_depth": [5, 15],
    "modelo__min_samples_split": [2, 15],
    "modelo__min_samples_leaf": [3, 8, 20],
    "modelo__max_features": ["sqrt", 0.5, None],
    "modelo__max_samples": [0.5, 1.0],
    "modelo__class_weight": [None, "balanced"]
}

In [0]:
# Hist Gradient Boosting

pipeline_hist_gradient = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ("modelo", HistGradientBoostingClassifier(random_state=42, early_stopping=True))
])

grid_hist_gradient = {
    "modelo__learning_rate": [0.03, 0.05, 0.1],
    "modelo__max_iter": [200, 400, 600],
    "modelo__max_leaf_nodes": [15, 31, 63],
    "modelo__min_samples_leaf": [10, 20, 50],
    "modelo__l2_regularization": [0.0, 0.1, 1.0]
}

In [0]:
# XG Boost

pipeline_xgboost = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ("modelo", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

grid_xgboost = {
    "modelo__n_estimators": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    "modelo__max_depth": [3, 6],
    "modelo__min_child_weight": [1, 5],
    "modelo__subsample": [0.8, 1.0],
    "modelo__colsample_bytree": [0.8, 1.0],
    "modelo__reg_alpha": [0.0, 0.1],
    "modelo__reg_lambda": [1.0, 5.0]
}

In [0]:
# Light GBM

pipeline_lightgbm = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ("modelo", LGBMClassifier(
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

grid_lightgbm = {
    "modelo__n_estimators": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    #"modelo__num_leaves": [31, 63],
    "modelo__max_depth": [6, 10],
    #"modelo__min_child_samples": [20, 50],
    #"modelo__subsample": [0.8, 1.0],
    #"modelo__colsample_bytree": [0.8, 1.0],
    #"modelo__reg_alpha": [0.0, 0.1],
    #"modelo__reg_lambda": [0.0, 1.0]
}

In [0]:
# Cat Boost

pipeline_catboost = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ("modelo", CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    ))
])

grid_catboost = {
    "modelo__iterations": [300, 600],
    "modelo__learning_rate": [0.03, 0.1],
    "modelo__depth": [4, 8],
    #"modelo__l2_leaf_reg": [3, 10],
    #"modelo__random_strength": [0.0, 1.0]
}

In [0]:
# Redes Neurais - MLP Classifier

pipeline_mlp = Pipeline([
    ('imputacao_0', imput_0),
    ('imput_max', imput_max_tail),
    # ('seletor_colunas', seletor_colunas),
    ('scaler', scaler),
    ('modelo', MLPClassifier(
        random_state=42,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

grid_mlp = {
    "modelo__hidden_layer_sizes": [(64,), (128, 64)],
    "modelo__activation": ["relu"],
    "modelo__alpha": [0.0001, 0.001],
    "modelo__learning_rate_init": [0.0005, 0.001],
    "modelo__batch_size": [128, 256]
}

In [0]:
# Dicionário dos Modelos

modelos = {
    "regressao_logistica": {
        "pipeline": pipeline_logistica,
        "param_grid": grid_logistica
    },
    "regressao_logistica_selecao_var": {
        "pipeline": pipeline_logistica_selecao,
        "param_grid": grid_logistica_selecao
    },
    "random_forest": {
        "pipeline": pipeline_random_forest,
        "param_grid": grid_random_forest
    },
    "hist_gradient_boosting": {
        "pipeline": pipeline_hist_gradient,
        "param_grid": grid_hist_gradient
    },
    "xgboost": {
        "pipeline": pipeline_xgboost,
        "param_grid": grid_xgboost
    },
    "lightgbm": {
        "pipeline": pipeline_lightgbm,
        "param_grid": grid_lightgbm
    },
    "catboost": {
        "pipeline": pipeline_catboost,
        "param_grid": grid_catboost
    },
    "mlp": {
        "pipeline": pipeline_mlp,
        "param_grid": grid_mlp
    }
}

In [0]:
# Configuração do GridSearch

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [0]:
# Configuração do MlFlow
os.environ["MLFLOW_DATABRICKS_TAGS_TO_SKIP"] = "extraContext"
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# VINCULA O SCRIPT PYTHON COM O EXPERIMENTO DO MLFLOW
mlflow.set_experiment(experiment_id=22108943160739)

mlflow.sklearn.autolog(
    log_models=False,
    max_tuning_runs=20
)

In [0]:
resultados = []
melhores_estimadores = {}

melhor_roc_auc = -np.inf
melhor_modelo_validacao = None
melhor_nome_modelo = None
melhor_run_id = None
melhores_parametros = None

for nome_modelo, configuracao in modelos.items():

    with mlflow.start_run(
        #run_name=f"selecao_{nome_modelo}"
        run_name=nome_modelo
    ) as run:

        grid = GridSearchCV(
            estimator=configuracao["pipeline"],
            param_grid=configuracao["param_grid"],
            scoring=scoring,
            refit="roc_auc",
            cv=cv,
            n_jobs=-1,
            verbose=2,
            return_train_score=True,
            error_score="raise"
        )

        grid.fit(X_train, y_train)

        melhor_estimador = grid.best_estimator_
        melhores_estimadores[nome_modelo] = melhor_estimador

        # Treino
        # Probabilidade da classe positiva
        y_train_prob = melhor_estimador.predict_proba(X_train)[:, 1]
        # Predição usando threshold padrão
        y_train_pred = (y_train_prob >= 0.5).astype(int)

        # Teste
        # Probabilidade da classe positiva
        y_test_prob = melhor_estimador.predict_proba(X_test)[:, 1]
        # Predição usando threshold padrão
        y_test_pred = (y_test_prob >= 0.5).astype(int)

        # OOT
        # Probabilidade da classe positiva
        y_oot_prob = melhor_estimador.predict_proba(df_oot[features])[:, 1]
        # Predição usando threshold padrão
        y_oot_pred = (y_oot_prob >= 0.5).astype(int)

        metricas = {
            "train_roc_auc": roc_auc_score(y_train, y_train_prob),
            "train_average_precision": average_precision_score(y_train, y_train_prob),
            "train_log_loss": log_loss(y_train, y_train_prob),
            "train_accuracy": accuracy_score(y_train, y_train_pred),
            "train_precision": precision_score(y_train, y_train_pred, zero_division=0),
            "train_recall": recall_score(y_train, y_train_pred, zero_division=0),
            "train_f1": f1_score(y_train, y_train_pred, zero_division=0),

            "test_roc_auc": roc_auc_score(y_test, y_test_prob),
            "test_average_precision": average_precision_score(y_test, y_test_prob),
            "test_log_loss": log_loss(y_test, y_test_prob),
            "test_accuracy": accuracy_score(y_test, y_test_pred),
            "test_precision": precision_score(y_test, y_test_pred, zero_division=0),
            "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_test_pred, zero_division=0),

            "oot_roc_auc": roc_auc_score(df_oot[target], y_oot_prob),
            "oot_average_precision": average_precision_score(df_oot[target], y_oot_prob),
            "oot_log_loss": log_loss(df_oot[target], y_oot_prob),
            "oot_accuracy": accuracy_score(df_oot[target], y_oot_pred),
            "oot_precision": precision_score(df_oot[target], y_oot_pred, zero_division=0),
            "oot_recall": recall_score(df_oot[target], y_oot_pred, zero_division=0),
            "oot_f1": f1_score(df_oot[target], y_oot_pred, zero_division=0)
        }

        mlflow.log_metrics(metricas)

        mlflow.log_params({
            "cv_best_roc_auc": grid.best_score_,
            "quantidade_features": X_train.shape[1],
            "quantidade_treino": X_train.shape[0],
            "quantidade_teste": X_test.shape[0],
            "taxa_churn_treino": float(y_train.mean()),
            "taxa_churn_teste": float(y_test.mean())
        })

        melhor_pipeline = grid.best_estimator_
        #features_finais = (melhor_pipeline[:-1].get_feature_names_out().tolist())
        features_finais = (melhor_pipeline[:-1].get_feature_names_out())
        #(((configuracao["pipeline"])[:-1]).get_feature_names_out())
        mlflow.log_params({
            "features_finais": features_finais
        })

        # Registra os melhores parâmetros do grid
        for parametro, valor in grid.best_params_.items():
            mlflow.log_param(
                f"best_{parametro}",
                valor
            )

        mlflow.set_tags({
            'features': features_finais,
        })

        input_example = X_train.head(5)

        signature = infer_signature(
            X_train,
            melhor_estimador.predict_proba(X_train)
        )

        mlflow.sklearn.log_model(
            sk_model=melhor_estimador,
            artifact_path="modelo_validacao",
            signature=signature,
            input_example=input_example
        )

        resultados.append({
            "modelo": nome_modelo,
            "cv_roc_auc": grid.best_score_,
            **metricas,
            "melhores_parametros": grid.best_params_,
            "run_id": run.info.run_id
        })

        if metricas["test_roc_auc"] > melhor_roc_auc:
            melhor_roc_auc = metricas["test_roc_auc"]
            melhor_modelo_validacao = melhor_estimador
            melhor_nome_modelo = nome_modelo
            melhor_run_id = run.info.run_id
            melhores_parametros = grid.best_params_.copy()

In [0]:
# Comparação dos Modelos

df_resultados = pd.DataFrame(resultados)

df_resultados = df_resultados.sort_values(
    by=[
        "test_roc_auc",
        "test_average_precision"
    ],
    ascending=False
).reset_index(drop=True)

colunas_resultado = [
    "modelo",
    "cv_roc_auc",
    "test_roc_auc",
    "test_average_precision",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_log_loss"
]

print(df_resultados[colunas_resultado])

In [0]:
df_resultados.to_csv(
    "comparacao_modelos_churn.csv",
    index=False
)

In [0]:
# Ranking na base de teste

# X_test["probabilidade_churn"] = (
    # melhor_modelo_validacao.predict_proba(X_test)[:, 1]
# )
# 
# X_test["churn_predito_05"] = (
    # X_test["probabilidade_churn"] >= 0.5
# ).astype(int)
# 
# ranking_teste = (
    # X_test.sort_values(
        # "probabilidade_churn",
        # ascending=False
    # )
    # .reset_index(drop=True)
# )
# 
# ranking_teste["posicao_ranking"] = (
    # np.arange(1, len(ranking_teste) + 1)
# )

In [0]:
# ranking_teste.head(10)

In [0]:
# Refit do melhor modelo com treino + teste

X_completo = pd.concat([X_train, X_test], axis=0)

y_completo = pd.concat([y_train, y_test], axis=0)

modelo_producao = clone(melhor_modelo_validacao)

modelo_producao.fit(X_completo, y_completo)

print("Modelo vencedor:", melhor_nome_modelo)
print("Quantidade de registros no refit:", len(X_completo))
print("Melhores parâmetros:")
print(melhores_parametros)

In [0]:
# Registro do modelo final em produção no MlFlow

with mlflow.start_run(
    run_name=f"producao_{melhor_nome_modelo}"
) as run_producao:

    mlflow.set_tag(
        "tipo_execucao",
        "refit_treino_mais_teste"
    )

    mlflow.set_tag(
        "modelo_selecionado",
        melhor_nome_modelo
    )

    mlflow.set_tag(
        "run_origem_validacao",
        melhor_run_id
    )

    mlflow.log_params({
        "modelo_vencedor": melhor_nome_modelo,
        "quantidade_refit": len(X_completo),
        "quantidade_features": X_completo.shape[1],
        "taxa_churn_refit": float(y_completo.mean()),
        "criterio_selecao": "test_roc_auc",
        "roc_auc_obtido_no_teste": float(
            melhor_roc_auc
        )
    })

    for parametro, valor in melhores_parametros.items():
        mlflow.log_param(
            f"best_{parametro}",
            valor
        )

    # Registra as métricas históricas de seleção
    linha_vencedora = df_resultados.loc[
        df_resultados["modelo"] == melhor_nome_modelo
    ].iloc[0]

    mlflow.log_metrics({
        "validation_roc_auc": float(
            linha_vencedora["test_roc_auc"]
        ),
        "validation_average_precision": float(
            linha_vencedora["test_average_precision"]
        )
    })

    input_example = X_completo.head(5)

    assinatura_producao = infer_signature(
        X_completo,
        modelo_producao.predict_proba(X_completo)
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=modelo_producao,
        artifact_path="modelo_producao",
        signature=assinatura_producao,
        input_example=input_example
    )

    run_id_producao = run_producao.info.run_id


In [0]:


feature_importance = pd.DataFrame({
    "variavel": X_train.columns,
    "importancia": melhor_modelo_validacao.named_steps['modelo'].feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importancia",
    ascending=False
)

print(feature_importance)